# Phase 3 — Bayesian Structural Recommendation Engine

**Build order (start simple, add structure):** this notebook is step 1–2 — scope the DAG, then build a *baseline* hierarchical model with country partial-pooling on the treatment effect. Later steps add country/year effects, the mechanism layer, and counterfactual simulation.

## Step 1 — Structural DAG (scope)

The end-state wants a *mechanism breakdown* ("tax X%, switching Y%, ..."), which a reduced-form single coefficient can't give. The full causal chain is:

```
policy  ->  energy prices  ->  fuel mix  ->  emissions
 HAVE        MISSING            HAVE          HAVE
```

- **policy:** `has_tax`, `has_ets`, `tax_price_only`, `ets_price_only`, `fuel_subsidy_gdp`
- **energy prices:** NOT measured (we have *carbon* prices = the policy lever, not the retail energy price firms/households face). This link **collapses** to reduced-form.
- **fuel mix:** `fossil_pct_filled`, `renewable_pct`, `nuclear_pct`, `energy_per_capita`, and emissions split by fuel (`coal/gas/oil_co2_per_capita`).
- **emissions:** `co2_per_capita_future_trend` (3-yr forward).

Realistic skeleton: a two-link chain `policy -> fuel mix -> emissions`, decomposed via the **Kaya identity** `CO2/pop = (GDP/pop) x (Energy/GDP) x (CO2/Energy)` — fuel-switching is the last term (carbon intensity of energy at fixed demand), which separates the *behavioural* mechanism from the emissions accounting identity. Price node deferred (multi-week data project, extra identification assumption).

## Step 2 — Baseline hierarchical model

Country partial-pooling on the treatment effect. **Deliberately NOT yet causal** — no country intercepts / year effects yet (added next). Purpose: learn and verify the hierarchical machinery.

$$y_i \sim \mathrm{Normal}(\alpha + \beta_{c[i]}\cdot \mathrm{has\_tax}_i,\ \sigma)$$
$$\beta_c \sim \mathrm{Normal}(\mu, \tau)$$

Priors (weakly-informative; centered at no-effect so the data, not us, moves $\mu$):
`alpha ~ Normal(0, 0.5)`, `mu ~ Normal(0, 0.5)`, `tau ~ HalfNormal(0.5)`, `sigma ~ HalfNormal(1.0)`.
Locations -> Normal; scales (SDs) -> positive-only HalfNormal.

In [1]:
import os
os.environ['PYTENSOR_FLAGS'] = 'cxx='   # PyTensor C backend broken on Windows/py3.13; nutpie compiles via numba

import pandas as pd
import numpy as np
import pymc as pm
import arviz as az

In [2]:
df = pd.read_csv('../data/cleaned/final_analysis_data.csv')

country_idx, country_labels = pd.factorize(df['country'])   # row -> int 0..162 (realizes the c[i] lookup)
n_countries = len(country_labels)
y   = df['co2_per_capita_future_trend'].values
tax = df['has_tax'].values

print(f"obs: {len(df)}  countries: {n_countries}  treated rows (has_tax): {int(tax.sum())}")

obs: 4218  countries: 163  treated rows (has_tax): 261


**Non-centered hierarchy.** Writing `beta ~ Normal(mu, tau)` directly creates *Neal's funnel* — the region the betas can occupy depends on `tau`, so the sampler stalls (`tau` got ESS=36, R-hat=1.08 in the centered version). The fix: sample a standardized `z ~ Normal(0,1)` and rebuild `beta = mu + tau*z`. Statistically identical (`mu + tau*z ~ Normal(mu, tau)`), but the geometry is decoupled so the sampler glides.

In [3]:
with pm.Model() as baseline:
    alpha = pm.Normal('alpha', mu=0, sigma=0.5)
    mu    = pm.Normal('mu',    mu=0, sigma=0.5)
    tau   = pm.HalfNormal('tau',   sigma=0.5)
    sigma = pm.HalfNormal('sigma', sigma=1.0)

    z     = pm.Normal('z', mu=0, sigma=1, shape=n_countries)   # non-centered
    beta  = pm.Deterministic('beta', mu + tau * z)             # beta_c = mu + tau*z_c ~ Normal(mu, tau)

    mu_i  = alpha + beta[country_idx] * tax
    y_obs = pm.Normal('y_obs', mu=mu_i, sigma=sigma, observed=y)

In [4]:
with baseline:
    idata = pm.sample(draws=1000, tune=1000, chains=4, target_accept=0.9,
                      random_seed=42, nuts_sampler='nutpie', progressbar=False)

NUTS[nutpie]: [alpha, mu, tau, sigma, z]


**Diagnostics first, interpretation second.** Want R-hat ≈ 1.00 (chains agree) and ESS in the hundreds+ (enough effectively-independent draws), with 0 divergences.

In [5]:
print("Convergence (top-level params):")
print(az.summary(idata, var_names=['alpha', 'mu', 'tau', 'sigma'], round_to=4).to_string())
print(f"\nDivergences: {int(idata.sample_stats['diverging'].sum())}")

post_mu = idata.posterior['mu'].values.flatten()
print(f"mu posterior mean: {post_mu.mean():.4f}   P(mu < 0): {(post_mu < 0).mean():.3f}")

Convergence (top-level params):


         mean      sd  eti89_lb  eti89_ub    ess_bulk   ess_tail   r_hat  mcse_mean  mcse_sd
alpha -0.0079  0.0054   -0.0165    0.0007   9271.2929  3084.2262  1.0001     0.0001   0.0000
mu    -0.1640  0.0305   -0.2125   -0.1166   3277.9440  2948.1450  1.0009     0.0005   0.0004
tau    0.0896  0.0387    0.0252    0.1535    693.1524   631.8018  1.0054     0.0014   0.0011
sigma  0.3482  0.0037    0.3421    0.3541  10202.2983  2956.4865  1.0012     0.0000   0.0000

Divergences: 0
mu posterior mean: -0.1640   P(mu < 0): 1.000


**Read.** All R-hat ≈ 1.00, ESS in the hundreds-to-thousands, 0 divergences — converged. `mu` ≈ −0.16 with `P(mu<0)` ≈ 1.0 echoes the Phase-1/2 ATT.

**Do NOT report this as causal yet.** The model has no country intercepts and no year effects, so `mu` absorbs cross-country level differences and global time trends, not just the policy. Next step: add country intercepts `alpha_c` (same partial-pooling trick on the baseline level) + year effects to recover a DiD-style causal estimate, then anchor priors to Phase 1/2 and splice in the mechanism layer.